# Cross-variable consistency, persistence benchmark and graph-attention ablation

Analyses of Sections 4.4 (persistence benchmark), 4.6 (cross-variable hydrological consistency, Figure 12) and 4.7
(ablation of the spatial graph attention module).

**Inputs**
- Daily test-period series written by `run_inference.py` (`inference_output/time_series/grid_<cell>_<run>.csv`;
  `pred_*` LandFormer, `target_*` VIC) for LandFormer-AR-All (`TAG`), LandFormer-NAR and the single-variable
  autoregressive models.
- ERA5 precipitation from `forcing.zarr` and VIC outputs from `Surr_obs/*.zarr`.
- Series of the model with graph attention, written by `analysis/graph_attention/gat_test_inference.py`.

Test period 27 September 2001 to 31 December 2006 (training 1990-1995, validation 1996-2000), held-out test cells and
an unseen parameter set. Units: ET and total runoff mm day⁻¹; SM volumetric water content of VIC layer 1 (m³ m⁻³);
P mm day⁻¹. Predictions are non-negative because the denormalised outputs are clipped at zero.
Results are written to `outputs/`.

In [ ]:
# Data locations (environment variables, see README)
import os as _os, sys as _sys
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.getcwd(), '..')))
from landformer_paths import DATA_DIR, EXT_DIR, RESULTS_DIR, ROOT  # noqa: E402


In [ ]:
import json, glob, re, warnings
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import spearmanr, ConstantInputWarning
warnings.filterwarnings("ignore", category=ConstantInputWarning)  # cells with a constant series give NaN
import matplotlib.pyplot as plt

ROOT = Path(f'{ROOT}')
OUT = Path(_os.getcwd()) / 'outputs'
OUT.mkdir(exist_ok=True)

TAG = 'Large_full_lag'
TS_DIR = ROOT / 'inference_output' / 'time_series'   # written by run_inference.py
GRID_METRICS = Path(f'{ROOT}/inference_output') / f'{TAG}_grid_metrics.csv'
FORCING = f'{DATA_DIR}/Surr_forcing/forcing.zarr'
Q_ZARR = f'{DATA_DIR}/Surr_obs/Q_real.zarr'
TEST_START, TEST_END = '2001-01-01', '2006-12-31'
PARAM_FILE = 130          # nfiles index of the test parameter set
WIN = 269                 # first predicted day index (270-day window)
VARS = {'ET': 'evap', 'SM': 'soil_moisture_total', 'Q': 'Q'}

## 0. Locate test cells and check the date mapping

`time_idx` k in the CSV is the k-th day from 2001-01-01. This cell re-reads VIC runoff for one cell from `Q_real.zarr`
and confirms that it equals `target_Q` in the CSV, which fixes both the dates and the parameter set.

In [ ]:
cells = pd.read_csv(GRID_METRICS)[['grid_idx', 'lat', 'lon']].set_index('grid_idx').sort_index()
print('test cells:', len(cells))

g0 = cells.index[0]
ts0 = pd.read_csv(TS_DIR / f'grid_{g0}_{TAG}.csv')
q = (xr.open_zarr(Q_ZARR).Q.isel(nfiles=PARAM_FILE)
       .sel(lat=cells.lat[g0], lon=cells.lon[g0], method='nearest')
       .sel(time=slice(TEST_START, TEST_END)).values)
print('days in CSV:', len(ts0), '| max |VIC zarr - CSV target_Q|:', np.abs(q[WIN:WIN + len(ts0)] - ts0.target_Q.values).max())
dates = pd.date_range(TEST_START, TEST_END)[WIN:]
print('predicted days:', dates[0].date(), 'to', dates[-1].date())

## 1. Precipitation for every test cell

Daily ERA5 precipitation on the same days as the predictions, nearest 0.0625° cell.

In [ ]:
fx = xr.open_zarr(FORCING)
prcp = fx.prcp.sel(time=slice(dates[0], dates[-1]))
ilat = np.abs(fx.lat.values[None, :] - cells.lat.values[:, None]).argmin(1)
ilon = np.abs(fx.lon.values[None, :] - cells.lon.values[:, None]).argmin(1)
print('max lat/lon offset (deg):', np.abs(fx.lat.values[ilat] - cells.lat.values).max(), np.abs(fx.lon.values[ilon] - cells.lon.values).max())
P_all = prcp.values                                   # (time, lat, lon)
P = P_all[:, ilat, ilon].T.astype('float64')          # (cell, time)
del P_all
assert P.shape == (len(cells), len(dates))
np.save(OUT / 'consistency_P_test_cells.npy', P)
print('P shape:', P.shape, '| median annual P (mm):', np.median(P.sum(1)) / (len(dates) / 365.25))

## 2. Per-cell diagnostics

For each test cell, with VIC and LandFormer computed by the same formula:

| Test | Quantity | Physical relation tested |
|---|---|---|
| T1 mass | ΣET/ΣP, ΣQ/ΣP, residual 1 − ΣET/ΣP − ΣQ/ΣP over 2001-09-27 to 2006-12-31 | partitioning of rainfall; VIC residual = storage change |
| T2 ET–SM | mean ET in each SM decile (decile edges from VIC SM of that cell) | moisture limitation of evaporation |
| T3 SM–Q | Spearman ρ between previous-day SM and runoff ratio Q/P on wet days (P ≥ 5 mm) | runoff generation depends on antecedent wetness |
| T4 timing | lag (0–60 d) of maximum cross-correlation P→SM and P→Q | response order and memory |
| T5 coupling | Pearson r of ET–SM, SM–Q, ET–Q; r(ET error, Q error) | joint behaviour, no compensation between heads |
| T6 bounds | % days at the ReLU floor (0) for LandFormer vs VIC zeros | physical limits |

Each model uses **its own** SM in T2–T3, so the test is internal consistency of the surrogate, not its SM accuracy.

In [ ]:
WET = 5.0        # mm/day threshold for wet days
MAXLAG = 60
NBIN = 10

def corr(a, b):
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan

def lag_of_max_xcorr(x, y, maxlag=MAXLAG):
    r = [corr(x[:len(x) - k], y[k:]) for k in range(maxlag + 1)]
    return int(np.nanargmax(r)) if np.isfinite(r).any() else np.nan

def per_cell(args):
    i, g = args
    df = pd.read_csv(TS_DIR / f'grid_{g}_{TAG}.csv')
    p = np.load(OUT / 'consistency_P_test_cells.npy', mmap_mode='r')[i]
    out = {'grid': g, 'sumP': p.sum()}
    edges = np.quantile(df.target_soil_moisture_total.values, np.linspace(0, 1, NBIN + 1))
    for src in ('vic', 'lf'):
        col = 'target' if src == 'vic' else 'pred'
        et, sm, qq = (df[f'{col}_{VARS[k]}'].to_numpy(float) for k in ('ET', 'SM', 'Q'))
        # T1
        out[f'{src}_ETP'] = et.sum() / p.sum()
        out[f'{src}_QP'] = qq.sum() / p.sum()
        out[f'{src}_resid'] = 1 - out[f'{src}_ETP'] - out[f'{src}_QP']
        # T2 (bins on VIC deciles so both curves share the x axis)
        b = np.clip(np.searchsorted(edges, sm, side='right') - 1, 0, NBIN - 1)
        for k in range(NBIN):
            m = b == k
            out[f'{src}_ETbin{k}'] = et[m].mean() if m.any() else np.nan
        # T3
        wet = p[1:] >= WET
        if wet.sum() >= 20:
            out[f'{src}_rho_SMant_RR'] = spearmanr(sm[:-1][wet], qq[1:][wet] / p[1:][wet]).statistic
        out[f'{src}_n_wet'] = int(wet.sum())
        # T4
        out[f'{src}_lag_P_SM'] = lag_of_max_xcorr(p, sm)
        out[f'{src}_lag_P_Q'] = lag_of_max_xcorr(p, qq)
        # T5
        out[f'{src}_r_ET_SM'] = corr(et, sm); out[f'{src}_r_SM_Q'] = corr(sm, qq); out[f'{src}_r_ET_Q'] = corr(et, qq)
        # T6
        for k in ('ET', 'SM', 'Q'):
            out[f'{src}_{k}_zero_pct'] = 100 * float((df[f'{col}_{VARS[k]}'].values == 0).mean())
    out['err_corr_ET_Q'] = corr(df.pred_evap.values - df.target_evap.values, df.pred_Q.values - df.target_Q.values)
    return out

with ProcessPoolExecutor(32) as ex:
    per = pd.DataFrame(list(ex.map(per_cell, list(enumerate(cells.index)), chunksize=50))).set_index('grid')
per = per.join(cells)
per.to_csv(OUT / 'consistency_per_cell_AR_All.csv')
print(per.shape)

### T1 — Long-term partitioning of precipitation

In [ ]:
def agree(v, l):
    d = l - v
    return {'VIC median': np.nanmedian(v), 'LandFormer median': np.nanmedian(l),
            'median (LF - VIC)': np.nanmedian(d), 'median |LF - VIC|': np.nanmedian(np.abs(d)),
            'r across cells': corr(v[np.isfinite(v) & np.isfinite(l)], l[np.isfinite(v) & np.isfinite(l)]),
            '% cells |LF - VIC| < 0.05': 100 * np.nanmean(np.abs(d) < 0.05)}

T1 = pd.DataFrame({k: agree(per[f'vic_{k}'].values, per[f'lf_{k}'].values) for k in ('ETP', 'QP', 'resid')}).T
T1.index = ['ET/P', 'Q/P', 'residual 1-ET/P-Q/P']
T1.round(3)

### T2 — ET as a function of soil moisture (moisture limitation)

In [ ]:
vic_curve = per[[f'vic_ETbin{k}' for k in range(NBIN)]].values
lf_curve = per[[f'lf_ETbin{k}' for k in range(NBIN)]].values
# per-cell RMS difference of the two curves, normalised by the VIC mean ET of the cell
nrmsd = np.sqrt(np.nanmean((lf_curve - vic_curve) ** 2, 1)) / np.nanmean(vic_curve, 1)
# shape: Spearman correlation between the two curves (same bins)
shape_r = np.array([spearmanr(a, b).statistic if np.isfinite(a).sum() > 3 and np.isfinite(b).sum() > 3 else np.nan
                    for a, b in zip(vic_curve, lf_curve)])
# direction: ET increases from driest to wettest decile
vic_up = vic_curve[:, -1] > vic_curve[:, 0]; lf_up = lf_curve[:, -1] > lf_curve[:, 0]
T2 = pd.Series({'median normalised RMS difference of ET-SM curves': np.nanmedian(nrmsd),
                'median Spearman r between curves': np.nanmedian(shape_r),
                '% cells where VIC ET rises with SM': 100 * vic_up.mean(),
                '% cells where LF ET rises with SM': 100 * lf_up.mean(),
                '% cells same direction': 100 * (vic_up == lf_up).mean()})
T2.round(3)

### T3 — Runoff ratio versus antecedent soil moisture

In [ ]:
v, l = per.vic_rho_SMant_RR, per.lf_rho_SMant_RR
ok = v.notna() & l.notna()
T3 = pd.Series({'cells with >= 20 wet days': int(ok.sum()),
                'VIC median rho(SM_t-1, Q/P)': v[ok].median(),
                'LF median rho(SM_t-1, Q/P)': l[ok].median(),
                '% cells VIC rho > 0': 100 * (v[ok] > 0).mean(),
                '% cells LF rho > 0': 100 * (l[ok] > 0).mean(),
                'r across cells (VIC vs LF rho)': corr(v[ok].values, l[ok].values)})
T3.round(3)

### T4 — Response timing

In [ ]:
T4 = pd.DataFrame({k: {'VIC median lag (d)': per[f'vic_{k}'].median(), 'LF median lag (d)': per[f'lf_{k}'].median(),
                        '% cells same lag': 100 * (per[f'vic_{k}'] == per[f'lf_{k}']).mean(),
                        '% cells |lag diff| <= 1 d': 100 * ((per[f'vic_{k}'] - per[f'lf_{k}']).abs() <= 1).mean()}
                   for k in ('lag_P_SM', 'lag_P_Q')}).T
T4.round(2)

### T5 — Coupling between variables and error coupling

In [ ]:
T5 = pd.DataFrame({k: {'VIC median r': per[f'vic_r_{k}'].median(), 'LF median r': per[f'lf_r_{k}'].median(),
                        'median per-cell gap (LF - VIC)': (per[f'lf_r_{k}'] - per[f'vic_r_{k}']).median()}
                   for k in ('ET_SM', 'SM_Q', 'ET_Q')}).T
print(T5.round(3))
print('\nmedian r(ET error, Q error):', round(per.err_corr_ET_Q.median(), 3))

### T6 — Physical bounds

In [ ]:
T6 = pd.DataFrame({k: {'VIC mean % days == 0': per[f'vic_{k}_zero_pct'].mean(),
                        'LF mean % days == 0 (ReLU floor)': per[f'lf_{k}_zero_pct'].mean(),
                        'LF median % days == 0': per[f'lf_{k}_zero_pct'].median()} for k in ('ET', 'SM', 'Q')}).T
T6.round(2)

### Figure 12

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 9))
for a, k, lab in [(ax[0, 0], 'ETP', 'ET / P'), (ax[0, 1], 'QP', 'Q / P')]:
    a.scatter(per[f'vic_{k}'], per[f'lf_{k}'], s=3, alpha=0.4)
    lim = [0, np.nanpercentile(per[[f'vic_{k}', f'lf_{k}']].values, 99.5)]
    a.plot(lim, lim, 'k--', lw=1); a.set_xlim(lim); a.set_ylim(lim)
    a.set_xlabel(f'VIC {lab}'); a.set_ylabel(f'LandFormer-AR-All {lab}')
x = np.arange(1, NBIN + 1)
ax[1, 0].plot(x, np.nanmedian(vic_curve, 0), 'o-', label='VIC')
ax[1, 0].plot(x, np.nanmedian(lf_curve, 0), 's--', label='LandFormer-AR-All')
ax[1, 0].set_xlabel('SM decile within cell (1 = driest)'); ax[1, 0].set_ylabel('median ET (mm/day)'); ax[1, 0].legend()
ax[1, 1].scatter(per.vic_rho_SMant_RR, per.lf_rho_SMant_RR, s=3, alpha=0.4)
ax[1, 1].plot([-1, 1], [-1, 1], 'k--', lw=1); ax[1, 1].set_xlim(-1, 1); ax[1, 1].set_ylim(-1, 1)
ax[1, 1].set_xlabel('VIC ρ(SM$_{t-1}$, Q/P), wet days'); ax[1, 1].set_ylabel('LandFormer ρ(SM$_{t-1}$, Q/P)')
for a, t in zip(ax.flat, 'abcd'):
    a.text(0.02, 1.02, f'({t})', transform=a.transAxes, fontweight='bold')
fig.tight_layout(); fig.savefig(OUT / 'consistency_consistency.png', dpi=200)

In [ ]:
res = {'model': TAG, 'checkpoint': f'{RESULTS_DIR}/Large_full_lag/models/best_model.pth',
       'period': [str(dates[0].date()), str(dates[-1].date())], 'n_cells': len(per), 'wet_threshold_mm': WET,
       'T1_partitioning': T1.to_dict(), 'T2_ET_SM': T2.to_dict(), 'T3_SM_runoff': T3.to_dict(),
       'T4_timing': T4.to_dict(), 'T5_coupling': T5.to_dict(), 'T5_err_corr_ET_Q': per.err_corr_ET_Q.median(),
       'T6_bounds': T6.to_dict()}
json.dump(res, open(OUT / 'consistency_consistency.json', 'w'), indent=2, default=float)
print('saved', OUT / 'consistency_consistency.json')

### Verification

(a) Recomputation of VIC ET/P and Q/P for 25 random cells directly from `evap.zarr` and `Q_real.zarr`.
(b) Origin of the unallocated precipitation of LandFormer: by aridity tercile and from days on which the predicted
runoff is zero. (c) Summary of the values reported in Section 4.6.

In [ ]:
EVAP_ZARR = f'{DATA_DIR}/Surr_obs/evap.zarr'
rng = np.random.default_rng(0)
pick = rng.choice(len(cells), 25, replace=False)
ev = xr.open_zarr(EVAP_ZARR).evap.isel(nfiles=PARAM_FILE).sel(time=slice(dates[0], dates[-1]))
qz = xr.open_zarr(Q_ZARR).Q.isel(nfiles=PARAM_FILE).sel(time=slice(dates[0], dates[-1]))
chk = []
for i in pick:
    g = cells.index[i]
    e = ev.sel(lat=cells.lat[g], lon=cells.lon[g], method='nearest').values
    q_ = qz.sel(lat=cells.lat[g], lon=cells.lon[g], method='nearest').values
    chk.append({'grid': g, 'dETP': abs(e.sum() / P[i].sum() - per.vic_ETP[g]), 'dQP': abs(q_.sum() / P[i].sum() - per.vic_QP[g])})
chk = pd.DataFrame(chk)
print('max |zarr - CSV| ET/P:', chk.dETP.max(), '| Q/P:', chk.dQP.max())

In [ ]:
per['resid_gap'] = per.lf_resid - per.vic_resid
per['aridity_tercile'] = pd.qcut(per.vic_ETP, 3, labels=['humid (low ET/P)', 'intermediate', 'arid (high ET/P)'])
by_arid = per.groupby('aridity_tercile', observed=True).agg(
    n=('resid_gap', 'size'), median_annual_P_mm=('sumP', lambda x: np.median(x) / (len(dates) / 365.25)),
    median_resid_gap=('resid_gap', 'median'), median_ETP_gap=('lf_ETP', 'median'))
by_arid['median_ETP_gap'] = per.assign(d=per.lf_ETP - per.vic_ETP).groupby('aridity_tercile', observed=True).d.median()
by_arid['median_QP_gap'] = per.assign(d=per.lf_QP - per.vic_QP).groupby('aridity_tercile', observed=True).d.median()
print(by_arid.round(3))

def clipped_share(args):
    i, g = args
    df = pd.read_csv(TS_DIR / f'grid_{g}_{TAG}.csv')
    p = np.load(OUT / 'consistency_P_test_cells.npy', mmap_mode='r')[i]
    z = df.pred_Q.values == 0
    deficit = (df.target_Q.values - df.pred_Q.values).sum()
    return {'grid': g, 'Q_deficit_P': deficit / p.sum(), 'Q_deficit_on_clipped_days_P': (df.target_Q.values[z] - 0).sum() / p.sum()}

with ProcessPoolExecutor(32) as ex:
    cs = pd.DataFrame(list(ex.map(clipped_share, list(enumerate(cells.index)), chunksize=50))).set_index('grid')
print('\nmedian total Q deficit (fraction of P):', round(cs.Q_deficit_P.median(), 4))
print('median VIC runoff on days LandFormer Q == 0 (fraction of P):', round(cs.Q_deficit_on_clipped_days_P.median(), 4))
print('share of the total deficit (sum over cells):', round(cs.Q_deficit_on_clipped_days_P.sum() / cs.Q_deficit_P.sum(), 3))

In [ ]:
quoted = {
 'ET/P LF / VIC':        (T1.loc['ET/P', 'LandFormer median'], T1.loc['ET/P', 'VIC median']),
 'Q/P LF / VIC':         (T1.loc['Q/P', 'LandFormer median'], T1.loc['Q/P', 'VIC median']),
 'r across cells ET/P, Q/P': (T1.loc['ET/P', 'r across cells'], T1.loc['Q/P', 'r across cells']),
 'residual LF / VIC':    (T1.loc['residual 1-ET/P-Q/P', 'LandFormer median'], T1.loc['residual 1-ET/P-Q/P', 'VIC median']),
 'ET-SM Spearman, NRMSD': (T2['median Spearman r between curves'], T2['median normalised RMS difference of ET-SM curves']),
 '% cells ET rises LF / VIC': (T2['% cells where LF ET rises with SM'], T2['% cells where VIC ET rises with SM']),
 'rho(SM_t-1, Q/P) LF / VIC': (T3['LF median rho(SM_t-1, Q/P)'], T3['VIC median rho(SM_t-1, Q/P)']),
 'coupling gaps ET-SM, SM-Q, ET-Q': tuple(T5['median per-cell gap (LF - VIC)']),
 'r(ET err, Q err)':     (per.err_corr_ET_Q.median(),),
 '% days zero Q, SM (LF)': (T6.loc['Q', 'LF mean % days == 0 (ReLU floor)'], T6.loc['SM', 'LF mean % days == 0 (ReLU floor)']),
 '% days zero Q, SM (VIC)': (T6.loc['Q', 'VIC mean % days == 0'], T6.loc['SM', 'VIC mean % days == 0']),
}
for k, v in quoted.items():
    print(f'{k:34s}', ', '.join(f'{x:.3f}' for x in v))
json.dump({'by_aridity': by_arid.reset_index().astype({'aridity_tercile': str}).to_dict('records'),
           'Q_deficit_median_fracP': cs.Q_deficit_P.median(),
           'Q_deficit_on_clipped_days_median_fracP': cs.Q_deficit_on_clipped_days_P.median(),
           'clipped_share_of_total_deficit': cs.Q_deficit_on_clipped_days_P.sum() / cs.Q_deficit_P.sum(),
           'zarr_check_max_abs': [chk.dETP.max(), chk.dQP.max()]},
          open(OUT / 'consistency_verification.json', 'w'), indent=2, default=float)

### Zero predictions

The targets are z-score normalised, and the denormalised outputs pass through a ReLU before the KGE loss is computed
(`UnifiedTrainer._denormalize_targets`). Predictions below zero therefore receive no gradient and are written as zero
at inference. This cell locates the days with zero predictions within the VIC distribution of each cell.

In [ ]:
def zero_days(args):
    i, g = args
    df = pd.read_csv(TS_DIR / f'grid_{g}_{TAG}.csv')
    out = {'grid': g}
    for k in ('Q', 'SM', 'ET'):
        v = df[f'target_{VARS[k]}'].values; z = df[f'pred_{VARS[k]}'].values == 0
        if z.any():
            pct = (np.argsort(np.argsort(v)) + 0.5) / len(v) * 100     # within-cell percentile of VIC
            out[f'{k}_vic_pct_on_zero_days'] = np.median(pct[z])
            out[f'{k}_share_zero_days_below_vic_p20'] = float((pct[z] <= 20).mean())
            out[f'{k}_vic_mean_on_zero_days'] = v[z].mean()
            out[f'{k}_vic_mean_all_days'] = v.mean()
    return out

with ProcessPoolExecutor(32) as ex:
    zd = pd.DataFrame(list(ex.map(zero_days, list(enumerate(cells.index)), chunksize=50))).set_index('grid')
WHY0 = pd.DataFrame({k: {'cells with any zero': int(zd[f'{k}_vic_pct_on_zero_days'].notna().sum()),
                          'median within-cell VIC percentile on zero days': zd[f'{k}_vic_pct_on_zero_days'].median(),
                          'median share of zero days in VIC lowest 20%': zd[f'{k}_share_zero_days_below_vic_p20'].median(),
                          'median VIC mean on zero days': zd[f'{k}_vic_mean_on_zero_days'].median(),
                          'median VIC mean on all days': zd[f'{k}_vic_mean_all_days'].median()} for k in ('Q', 'SM', 'ET')}).T
WHY0.to_csv(OUT / 'consistency_zero_days.csv')
WHY0.round(3)

## Ablation of the spatial graph attention module (Section 4.7)

LandFormer-AR-All without and with the graph attention module, trained with identical cells, parameter sets, period,
inputs and hyperparameters; the model with the module is trained on mini-batches of neighbouring cells of the same
day. Both are evaluated on the same test cells, parameter set and period. Per-cell skill is compared with the
Wilcoxon signed-rank test and the rank-biserial correlation.

In [ ]:
warnings.filterwarnings('ignore', category=RuntimeWarning)
GAT_DIR = ROOT / 'analysis' / 'graph_attention' / 'outputs'
GAT_TAG = 'GAT'   # tag passed to gat_test_inference.py
info = json.load(open(GAT_DIR / f'{GAT_TAG}_run_info.json'))
print('GAT checkpoint:', info['checkpoint'], '| test graph:', info['graph_stats'])

def metrics(o, s):
    o = np.asarray(o, float); s = np.asarray(s, float)
    r = corr(s, o); a = s.std() / o.std() if o.std() > 0 else np.nan; b = s.mean() / o.mean() if o.mean() != 0 else np.nan
    return {'kge': 1 - np.sqrt((r - 1) ** 2 + (a - 1) ** 2 + (b - 1) ** 2), 'nse': 1 - ((s - o) ** 2).sum() / ((o - o.mean()) ** 2).sum(),
            'r': r, 'rmse': np.sqrt(((s - o) ** 2).mean())}

def pair_cell(g):
    a = pd.read_csv(TS_DIR / f'grid_{g}_{TAG}.csv')
    b = pd.read_csv(GAT_DIR / 'time_series' / f'grid_{g}_{GAT_TAG}.csv')
    m = a.merge(b, on='time_idx', suffixes=('_a', '_g'))
    out = {'grid': g, 'n_days': len(m),
           'max_target_diff': max(float(np.abs(m[f'target_{v}_a'] - m[f'target_{v}_g']).max()) for v in VARS.values())}
    for k, v in VARS.items():
        for tag, suf in (('arall', '_a'), ('gat', '_g')):
            for mk, mv in metrics(m[f'target_{v}_a'], m[f'pred_{v}{suf}']).items():
                out[f'{tag}_{k}_{mk}'] = mv
    return out

gat_cells = sorted(int(f.name.split('_')[1]) for f in (GAT_DIR / 'time_series').glob(f'grid_*_{GAT_TAG}.csv'))
both = sorted(set(gat_cells) & set(cells.index))
with ProcessPoolExecutor(32) as ex:
    r15 = pd.DataFrame(list(ex.map(pair_cell, both, chunksize=50))).set_index('grid')
print('cells compared:', len(r15), '| days per cell:', r15.n_days.unique(), '| max |VIC target difference|:', r15.max_target_diff.max())
r15.to_csv(OUT / 'gat_ablation_per_cell.csv')

In [ ]:
from scipy.stats import wilcoxon
rows = []
for k in VARS:
    for mk, better in (('kge', 1), ('nse', 1), ('r', 1), ('rmse', -1)):
        a, g = r15[f'arall_{k}_{mk}'], r15[f'gat_{k}_{mk}']
        ok = np.isfinite(a) & np.isfinite(g)          # cells with constant series give inf/NaN
        d = (g - a)[ok]
        stat = wilcoxon(g[ok], a[ok])
        rbc = (np.sum(np.sign(d) == np.sign(better)) - np.sum(np.sign(d) == -np.sign(better))) / max(1, np.sum(d != 0))
        rows.append({'variable': k, 'metric': mk.upper(), 'n cells': int(ok.sum()), 'AR-All median': a[ok].median(), 'AR-All + GAT median': g[ok].median(),
                     'median per-cell difference': d.median(), '% cells GAT better': 100 * np.mean(np.sign(d) == np.sign(better)),
                     'Wilcoxon p': stat.pvalue, 'rank-biserial r (GAT better > 0)': rbc})
R15 = pd.DataFrame(rows)
R15.to_csv(OUT / 'gat_ablation_summary.csv', index=False)
R15.round(4)

## Persistence benchmark (Section 4.4)

During evaluation the lagged inputs hold the VIC values of the previous day, so the autoregressive results measure
one-step-ahead skill given the previous-day state. Persistence, ŷ(t) = y(t−1), uses the same information without a
model. Each model is compared with persistence on its own test cells.

In [ ]:
NAR_DIR = ROOT / 'inference_output/time_series'
SETS = {'Set 1 (AR-All cells)': [('AR-All (ET+Q lags)', TS_DIR, 'Large_full_lag'), ('NAR 109M (no lags)', NAR_DIR, 'TEST_HYDRODL_4')],
        'Set 2 (single-lag cells)': [('AR-ET', TS_DIR, 'Lagged_E'), ('AR-Q', TS_DIR, 'Lagged_Q'), ('AR-SM', TS_DIR, 'Lagged_SM')]}

def cell_list(d, tag):
    return sorted(int(f.name.split('_')[1]) for f in d.glob(f'grid_*_{tag}.csv'))

def persist_cell(args):
    g, models = args
    out = {'grid': g}
    ref = None
    for name, d, tag in models:
        df = pd.read_csv(d / f'grid_{g}_{tag}.csv').sort_values('time_idx')
        if ref is None:
            ref = df
            for k, v in VARS.items():
                o = df[f'target_{v}'].to_numpy(float)
                for mk, mv in metrics(o[1:], o[:-1]).items():      # persistence: y_t = y_{t-1}
                    out[f'Persistence|{k}|{mk}'] = mv
        else:
            assert np.allclose(df.target_Q.values, ref.target_Q.values), (g, tag)
        for k, v in VARS.items():
            o = df[f'target_{v}'].to_numpy(float)[1:]; s_ = df[f'pred_{v}'].to_numpy(float)[1:]
            for mk, mv in metrics(o, s_).items():
                out[f'{name}|{k}|{mk}'] = mv
    return out

R16 = {}
for set_name, models in SETS.items():
    common = sorted(set.intersection(*[set(cell_list(d, tag)) for _, d, tag in models]))
    with ProcessPoolExecutor(32) as ex:
        R16[set_name] = pd.DataFrame(list(ex.map(persist_cell, [(g, models) for g in common], chunksize=50))).set_index('grid')
    print(set_name, len(common), 'cells')

In [ ]:
rows = []
for set_name, df in R16.items():
    names = ['Persistence'] + [m[0] for m in SETS[set_name]]
    for k in VARS:
        for mk in ('kge', 'nse', 'rmse'):
            row = {'set': set_name, 'variable': k, 'metric': mk.upper()}
            for n in names:
                v = df[f'{n}|{k}|{mk}']; row[n] = v[np.isfinite(v)].median()
            for n in names[1:]:
                a, p_ = df[f'{n}|{k}|{mk}'], df[f'Persistence|{k}|{mk}']
                ok = np.isfinite(a) & np.isfinite(p_)
                better = (a[ok] < p_[ok]) if mk == 'rmse' else (a[ok] > p_[ok])
                row[f'% cells {n} beats persistence'] = 100 * better.mean()
            rows.append(row)
R16S = pd.DataFrame(rows)
R16S.to_csv(OUT / 'persistence_persistence.csv', index=False)
pd.set_option('display.width', 250)
R16S.round(3)